In [1]:
print("step 1")
import numpy as np
print("step 2")
import pandas as pd
print("step 3")

step 1
step 2
step 3


In [3]:
"""
Notebook 03 — Feature engineering, baseline model, first XGBoost.

Цель: построить базовые фичи на обучающем датасете, обучить baseline и 
первый XGBoost, сравнить в MLflow.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

# Подавить предупреждения, мешающие читать вывод
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

# Пути
PROCESSED_DIR = Path("../data/processed")

# Загрузка финального training set
df = pd.read_parquet(PROCESSED_DIR / "training_set_v1.parquet")
print(f"Training set: {len(df):,} строк, {len(df.columns)} колонок")
print(f"Колонки: {list(df.columns)}")
print()
print(f"Период: {df['incident_hour'].min()} — {df['incident_hour'].max()}")
print(f"Уникальных инцидентов: {df['event_id'].nunique():,}")
print(f"Уникальных маршрутов: {df['bus_route'].nunique():,}")
print()
print("Распределение target (uplift_t1):")
print(df["uplift_t1"].describe([0.05, 0.5, 0.95]))

Training set: 1,394,946 строк, 14 колонок
Колонки: ['event_id', 'incident_hour', 'bus_route', 'actual_t0', 'actual_t1', 'baseline_t0', 'baseline_t1', 'uplift_t0', 'uplift_t1', 'status_label', 'num_lines_affected', 'n_boroughs_affected', 'lines_affected_str', 'boroughs_affected_str']

Период: 2024-10-01 00:00:00 — 2024-12-31 22:00:00
Уникальных инцидентов: 4,762
Уникальных маршрутов: 384

Распределение target (uplift_t1):
count    1.394946e+06
mean     1.591284e+00
std      4.208556e+01
min     -2.320692e+03
5%      -3.521429e+01
50%      0.000000e+00
95%      5.146154e+01
max      6.564615e+02
Name: uplift_t1, dtype: float64


In [4]:
# === Feature engineering ===

# Делаем копию, чтобы не модифицировать исходные данные
df_feat = df.copy()

# === 1. Временные признаки ===
df_feat["hour"] = df_feat["incident_hour"].dt.hour
df_feat["day_of_week"] = df_feat["incident_hour"].dt.dayofweek  # 0=пн, 6=вс
df_feat["is_weekend"] = (df_feat["day_of_week"] >= 5).astype(int)
df_feat["month"] = df_feat["incident_hour"].dt.month
df_feat["day_of_month"] = df_feat["incident_hour"].dt.day

# Время суток в категориях (для интерпретируемости)
def time_of_day(h):
    if 6 <= h < 10:
        return "morning_rush"     # утренний час пик
    elif 10 <= h < 16:
        return "midday"           # дневное время
    elif 16 <= h < 20:
        return "evening_rush"     # вечерний час пик
    elif 20 <= h < 24:
        return "evening"          # вечер
    else:
        return "night"            # ночь

df_feat["time_of_day"] = df_feat["hour"].apply(time_of_day)

# === 2. Признаки инцидента ===
# Простой статус (берём первый токен до " | ")
df_feat["status_main"] = df_feat["status_label"].str.split(" | ").str[0]

# Считаем основные категории, чтобы знать что у нас в фиче
print("Распределение time_of_day:")
print(df_feat["time_of_day"].value_counts())
print()
print("Распределение status_main (топ-10):")
print(df_feat["status_main"].value_counts().head(10))
print()

# === 3. Признак "это родной borough маршрута" ===
# Если в зоне инцидента только один borough и маршрут оттуда → это самый сильный сигнал
# Иначе — у инцидента несколько boroughs

# Делаем reverse: для каждого маршрута его borough
def get_route_main_borough(route):
    """Главный borough маршрута (без express)."""
    if pd.isna(route):
        return "unknown"
    if route.startswith("BX"):
        return "Bronx"
    if route.startswith("BM"):
        return "Brooklyn"
    if route.startswith("Q"):
        return "Queens"
    if route.startswith("B"):
        return "Brooklyn"
    if route.startswith("M"):
        return "Manhattan"
    if route.startswith("S"):
        return "Staten Island"
    if route.startswith("X"):
        return "Manhattan"
    return "unknown"

df_feat["route_borough"] = df_feat["bus_route"].apply(get_route_main_borough)
df_feat["is_express"] = df_feat["bus_route"].str.startswith(
    ("QM", "SIM", "BXM", "BM", "X")
).astype(int)

# Маршрут расположен в одном из затронутых boroughs?
def borough_match(row):
    boroughs = set(row["boroughs_affected_str"].split(" | "))
    return int(row["route_borough"] in boroughs)

df_feat["route_in_zone"] = df_feat.apply(borough_match, axis=1)

print("Распределение route_in_zone:")
print(df_feat["route_in_zone"].value_counts())
print()

print(f"Финальный датасет: {df_feat.shape}")
print(f"Новые колонки: {[c for c in df_feat.columns if c not in df.columns]}")

Распределение time_of_day:
time_of_day
midday          384053
evening         310690
night           289120
evening_rush    257381
morning_rush    153702
Name: count, dtype: int64

Распределение status_main (топ-10):
status_main
delays              1226975
stops-skipped         81789
express-to-local      37158
reroute               31696
part-suspended        10839
boarding-change        3974
special-notice         1044
suspended               989
severe-delays           276
cancellations           206
Name: count, dtype: int64

Распределение route_in_zone:
route_in_zone
1    1148633
0     246313
Name: count, dtype: int64

Финальный датасет: (1394946, 24)
Новые колонки: ['hour', 'day_of_week', 'is_weekend', 'month', 'day_of_month', 'time_of_day', 'status_main', 'route_borough', 'is_express', 'route_in_zone']


In [5]:
print("Распределение route_in_zone:")
print(df_feat["route_in_zone"].value_counts())
print()
print(f"Доля match: {df_feat['route_in_zone'].mean():.1%}")
print()
print("Sample строк со всеми ключевыми фичами:")
df_feat[["bus_route", "route_borough", "boroughs_affected_str", 
         "route_in_zone", "is_express", "status_main", 
         "time_of_day", "uplift_t1"]].head(10)

Распределение route_in_zone:
route_in_zone
1    1148633
0     246313
Name: count, dtype: int64

Доля match: 82.3%

Sample строк со всеми ключевыми фичами:


,bus_route,route_borough,boroughs_affected_str,route_in_zone,is_express,status_main,time_of_day,uplift_t1
0,B1,Brooklyn,Bronx | Brooklyn | Manhattan,1,0,part-suspended,night,1.000000
1,B100,Brooklyn,Bronx | Brooklyn | Manhattan,1,0,part-suspended,night,-0.571429
2,B101,Brooklyn,Bronx | Brooklyn | Manhattan,1,0,part-suspended,night,0.000000
3,B103,Brooklyn,Bronx | Brooklyn | Manhattan,1,0,part-suspended,night,1.857143
4,B106,Brooklyn,Bronx | Brooklyn | Manhattan,1,0,part-suspended,night,0.000000
5,B11,Brooklyn,Bronx | Brooklyn | Manhattan,1,0,part-suspended,night,-1.357143
6,B111,Brooklyn,Bronx | Brooklyn | Manhattan,1,0,part-suspended,night,0.000000
7,B12,Brooklyn,Bronx | Brooklyn | Manhattan,1,0,part-suspended,night,-2.142857
8,B13,Brooklyn,Bronx | Brooklyn | Manhattan,1,0,part-suspended,night,2.642857
9,B14,Brooklyn,Bronx | Brooklyn | Manhattan,1,0,part-suspended,night,1.428571


In [6]:
# === Train/test split по времени ===

# Граница: всё до 1 декабря — train, всё с 1 декабря — test
SPLIT_DATE = pd.Timestamp("2024-12-01")

train_mask = df_feat["incident_hour"] < SPLIT_DATE
test_mask = df_feat["incident_hour"] >= SPLIT_DATE

train_df = df_feat[train_mask].copy()
test_df = df_feat[test_mask].copy()

print(f"Train: {len(train_df):,} строк ({len(train_df)/len(df_feat):.1%})")
print(f"  Период: {train_df['incident_hour'].min()} — {train_df['incident_hour'].max()}")
print(f"  Уникальных инцидентов: {train_df['event_id'].nunique():,}")
print()
print(f"Test:  {len(test_df):,} строк ({len(test_df)/len(df_feat):.1%})")
print(f"  Период: {test_df['incident_hour'].min()} — {test_df['incident_hour'].max()}")
print(f"  Уникальных инцидентов: {test_df['event_id'].nunique():,}")
print()

# Удаляем строки с NaN в target — нечего обучать/тестировать
train_df = train_df.dropna(subset=["uplift_t1"])
test_df = test_df.dropna(subset=["uplift_t1"])

print(f"После удаления NaN в target:")
print(f"  Train: {len(train_df):,}")
print(f"  Test:  {len(test_df):,}")
print()

# Распределение target в каждом сете — должно быть похожим (без сдвига)
print("Mean uplift_t1:")
print(f"  Train: {train_df['uplift_t1'].mean():.3f}")
print(f"  Test:  {test_df['uplift_t1'].mean():.3f}")
print()
print("Std uplift_t1:")
print(f"  Train: {train_df['uplift_t1'].std():.3f}")
print(f"  Test:  {test_df['uplift_t1'].std():.3f}")

Train: 836,875 строк (60.0%)
  Период: 2024-10-01 00:00:00 — 2024-11-17 23:00:00
  Уникальных инцидентов: 2,852

Test:  558,071 строк (40.0%)
  Период: 2024-12-01 00:00:00 — 2024-12-31 22:00:00
  Уникальных инцидентов: 1,910

После удаления NaN в target:
  Train: 836,875
  Test:  558,071

Mean uplift_t1:
  Train: 7.509
  Test:  -7.283

Std uplift_t1:
  Train: 33.474
  Test:  51.143


In [7]:
# Сколько примеров было ДО удаления NaN, по дням
before_drop = df_feat.groupby(df_feat["incident_hour"].dt.date).size()
print("Примеров по дням (до dropna):")
print(before_drop.describe())
print()
print("Min день:", before_drop.idxmin(), "→", before_drop.min(), "примеров")
print("Max день:", before_drop.idxmax(), "→", before_drop.max(), "примеров")
print()

# Сколько NaN в uplift_t1 по дням
nan_by_day = df_feat.groupby(df_feat["incident_hour"].dt.date)["uplift_t1"].apply(
    lambda s: s.isna().sum()
)
print("Дни с наибольшим числом NaN в uplift_t1:")
print(nan_by_day.sort_values(ascending=False).head(20))

Примеров по дням (до dropna):
count       78.000000
mean     17883.923077
std       4224.096229
min       7908.000000
25%      15486.250000
50%      18143.500000
75%      20387.750000
max      27784.000000
dtype: float64

Min день: 2024-11-02 → 7908 примеров
Max день: 2024-10-18 → 27784 примеров

Дни с наибольшим числом NaN в uplift_t1:
incident_hour
2024-10-01    0
2024-12-03    0
2024-12-10    0
2024-12-09    0
2024-12-08    0
2024-12-07    0
2024-12-06    0
2024-12-05    0
2024-12-04    0
2024-12-02    0
2024-12-12    0
2024-12-01    0
2024-11-17    0
2024-11-16    0
2024-11-15    0
2024-11-14    0
2024-11-13    0
2024-11-12    0
2024-12-11    0
2024-12-13    0
Name: uplift_t1, dtype: int64


In [8]:
# Среднее uplift_t1 по неделям
df_feat["week"] = df_feat["incident_hour"].dt.isocalendar().week
weekly_target = df_feat.groupby("week")["uplift_t1"].agg(["mean", "std", "count"])
print("Mean target по неделям:")
print(weekly_target)

Mean target по неделям:
           mean        std   count
week                              
1    -28.672794  73.260866   34249
40     4.464900  33.307096   95181
41    11.728012  32.970708  122536
42     4.186114  38.106980  136621
43    10.620465  30.464982  128346
44     9.053234  34.312780   89676
45     7.024215  31.938385  131006
46     5.656917  31.883199  133509
48    -6.694545  21.002655   12722
49     4.155512  25.374473  132032
50     3.124445  27.925222  125962
51    -0.219323  25.582675  135305
52   -33.191677  85.744066  117801


In [9]:
# === Корректный train/test split с исключением аномальных периодов ===

TRAIN_START = pd.Timestamp("2024-10-01")
TRAIN_END   = pd.Timestamp("2024-11-18")  # train: 1 окт – 17 ноя включительно
TEST_START  = pd.Timestamp("2024-12-01")
TEST_END    = pd.Timestamp("2024-12-23")  # test: 1 – 22 дек включительно

# Маски
train_mask = (df_feat["incident_hour"] >= TRAIN_START) & (df_feat["incident_hour"] < TRAIN_END)
test_mask  = (df_feat["incident_hour"] >= TEST_START) & (df_feat["incident_hour"] < TEST_END)

train_df = df_feat[train_mask].dropna(subset=["uplift_t1"]).copy()
test_df  = df_feat[test_mask].dropna(subset=["uplift_t1"]).copy()

# Период перерыва
gap_mask = (df_feat["incident_hour"] >= TRAIN_END) & (df_feat["incident_hour"] < TEST_START)
holiday_mask = df_feat["incident_hour"] >= TEST_END

excluded_gap = df_feat[gap_mask].shape[0]
excluded_holiday = df_feat[holiday_mask].shape[0]

print("=" * 60)
print("Train/test split")
print("=" * 60)
print(f"Train:  {len(train_df):>8,} строк "
      f"({train_df['incident_hour'].min().date()} — {train_df['incident_hour'].max().date()})")
print(f"        {train_df['event_id'].nunique():,} уникальных инцидентов")
print()
print(f"Test:   {len(test_df):>8,} строк "
      f"({test_df['incident_hour'].min().date()} — {test_df['incident_hour'].max().date()})")
print(f"        {test_df['event_id'].nunique():,} уникальных инцидентов")
print()
print(f"Исключено:")
print(f"  Gap (18-30 ноя, дыра в данных):     {excluded_gap:>8,}")
print(f"  Holidays (23-31 дек):               {excluded_holiday:>8,}")
print()
print("=" * 60)
print("Сравнение распределений target (uplift_t1)")
print("=" * 60)
print(f"           Mean      Std     Median     5%      95%")
print(f"Train:    {train_df['uplift_t1'].mean():>6.2f}   {train_df['uplift_t1'].std():>6.2f}   "
      f"{train_df['uplift_t1'].median():>5.1f}   "
      f"{train_df['uplift_t1'].quantile(0.05):>6.1f}   {train_df['uplift_t1'].quantile(0.95):>6.1f}")
print(f"Test:     {test_df['uplift_t1'].mean():>6.2f}   {test_df['uplift_t1'].std():>6.2f}   "
      f"{test_df['uplift_t1'].median():>5.1f}   "
      f"{test_df['uplift_t1'].quantile(0.05):>6.1f}   {test_df['uplift_t1'].quantile(0.95):>6.1f}")

Train/test split
Train:   836,875 строк (2024-10-01 — 2024-11-17)
        2,852 уникальных инцидентов

Test:    406,021 строк (2024-12-01 — 2024-12-22)
        1,390 уникальных инцидентов

Исключено:
  Gap (18-30 ноя, дыра в данных):            0
  Holidays (23-31 дек):                152,050

Сравнение распределений target (uplift_t1)
           Mean      Std     Median     5%      95%
Train:      7.51    33.47     0.0    -18.4     61.7
Test:       2.04    26.25     0.0    -28.8     40.3


In [10]:
# === Готовим X и y для обучения ===

# Фичи модели
NUMERIC_FEATURES = [
    "hour", "day_of_week", "is_weekend", "month", "day_of_month",
    "num_lines_affected", "n_boroughs_affected",
    "is_express", "route_in_zone",
    "baseline_t0", "baseline_t1",  # базовый профиль как фича — модель учится поправлять
    "actual_t0",  # фактический поток в час инцидента — сильный предиктор для t+1
]

CATEGORICAL_FEATURES = [
    "time_of_day", "status_main", "route_borough"
]

TARGET = "uplift_t1"

print("Numeric features:", NUMERIC_FEATURES)
print("Categorical features:", CATEGORICAL_FEATURES)
print(f"Target: {TARGET}")
print()

# One-hot encoding для категориальных (XGBoost умеет работать с числовыми + dummies)
X_train = pd.get_dummies(
    train_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES],
    columns=CATEGORICAL_FEATURES,
    drop_first=False
)
X_test = pd.get_dummies(
    test_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES],
    columns=CATEGORICAL_FEATURES,
    drop_first=False
)

# Выравниваем колонки (если в test нет какой-то категории — добавляем нулевую)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train_df[TARGET]
y_test = test_df[TARGET]

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")
print()
print(f"Всего фичей после one-hot: {X_train.shape[1]}")
print(f"Имена фичей: {list(X_train.columns)}")

Numeric features: ['hour', 'day_of_week', 'is_weekend', 'month', 'day_of_month', 'num_lines_affected', 'n_boroughs_affected', 'is_express', 'route_in_zone', 'baseline_t0', 'baseline_t1', 'actual_t0']
Categorical features: ['time_of_day', 'status_main', 'route_borough']
Target: uplift_t1

X_train shape: (836875, 32)
X_test shape:  (406021, 32)
y_train shape: (836875,)
y_test shape:  (406021,)

Всего фичей после one-hot: 32
Имена фичей: ['hour', 'day_of_week', 'is_weekend', 'month', 'day_of_month', 'num_lines_affected', 'n_boroughs_affected', 'is_express', 'route_in_zone', 'baseline_t0', 'baseline_t1', 'actual_t0', 'time_of_day_evening', 'time_of_day_evening_rush', 'time_of_day_midday', 'time_of_day_morning_rush', 'time_of_day_night', 'status_main_boarding-change', 'status_main_cancellations', 'status_main_delays', 'status_main_express-to-local', 'status_main_part-suspended', 'status_main_reroute', 'status_main_severe-delays', 'status_main_special-notice', 'status_main_stops-skipped', 's

In [11]:
import time
from pathlib import Path

import numpy as np
import mlflow
import mlflow.xgboost
import mlflow.sklearn
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ---- 1. Настройка MLflow -----------------------------------------------
# mlruns должен лежать в корне проекта, а не внутри notebooks/
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
mlflow.set_tracking_uri(f"file:{(project_root / 'mlruns').as_posix()}")
mlflow.set_experiment("metro_bus_uplift_v1")
print(f"MLflow tracking URI : {mlflow.get_tracking_uri()}")
print(f"Эксперимент         : metro_bus_uplift_v1")

# ---- 2. Утилита для метрик ---------------------------------------------
def calc_metrics(y_true, y_pred):
    """MAE, RMSE и устойчивый MAPE.
    
    MAPE классический у нас неприменим, т.к. медиана target = 0
    (любое деление на ноль даёт inf). Поэтому считаем MAPE
    только на наблюдениях, где |y_true| > 1 пассажир.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    mask = np.abs(y_true) > 1.0
    if mask.sum() > 0:
        mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    else:
        mape = np.nan
    
    return mae, rmse, mape, int(mask.sum())

# ---- 3. Защита от рассинхрона колонок train/test -----------------------
# Если one-hot encoding делался на train и test раздельно, колонки могут
# различаться. Выравниваем test по train (отсутствующие колонки = 0).
missing_in_test = set(X_train.columns) - set(X_test.columns)
extra_in_test   = set(X_test.columns)  - set(X_train.columns)
if missing_in_test or extra_in_test:
    print(f"⚠️  Найдено расхождение колонок: "
          f"в test нет {len(missing_in_test)}, лишних {len(extra_in_test)}")
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
    print("   → X_test выровнен по X_train")
else:
    print("✅ Колонки X_train и X_test совпадают")

print(f"\nX_train shape: {X_train.shape}")
print(f"X_test  shape: {X_test.shape}")

# ---- 4. BASELINE: предсказание uplift = 0 -----------------------------
# Это и есть «прогноз по историческому среднему»: мы говорим, что
# инцидент не меняет привычный поток (baseline_t1). Любая ML-модель
# должна побеждать этот наивный прогноз — иначе она не нужна.
y_pred_baseline = np.zeros(len(y_test))
mae_b, rmse_b, mape_b, n_mape_b = calc_metrics(y_test.values, y_pred_baseline)

print("\n" + "=" * 55)
print("BASELINE (predict uplift = 0)")
print("=" * 55)
print(f"MAE  : {mae_b:.3f} пасс/час")
print(f"RMSE : {rmse_b:.3f} пасс/час")
print(f"MAPE : {mape_b:.2f}%  (на {n_mape_b:,} наблюдениях, где |y|>1)")

with mlflow.start_run(run_name="baseline_zero_uplift"):
    mlflow.log_params({
        "model_type":   "baseline",
        "strategy":     "predict_zero_uplift",
        "n_train":      len(y_train),
        "n_test":       len(y_test),
        "n_features":   X_train.shape[1],
        "train_period": "2024-10-01 .. 2024-11-17",
        "test_period":  "2024-12-01 .. 2024-12-22",
    })
    mlflow.log_metrics({
        "mae":  mae_b,
        "rmse": rmse_b,
        "mape": mape_b if not np.isnan(mape_b) else -1.0,
        "mape_n_samples": n_mape_b,
    })
    print("\n✅ Baseline залогирован в MLflow")

2026/04/25 19:55:11 INFO mlflow.tracking.fluent: Experiment with name 'metro_bus_uplift_v1' does not exist. Creating a new experiment.


MLflow tracking URI : file:c:/metro-bus-ml/mlruns
Эксперимент         : metro_bus_uplift_v1
✅ Колонки X_train и X_test совпадают

X_train shape: (836875, 32)
X_test  shape: (406021, 32)

BASELINE (predict uplift = 0)
MAE  : 11.678 пасс/час
RMSE : 26.330 пасс/час
MAPE : 100.00%  (на 222,888 наблюдениях, где |y|>1)

✅ Baseline залогирован в MLflow


In [12]:
import os
from mlflow.tracking import MlflowClient

# ---- A. Установить переменные окружения для будущих run'ов -------------
# MLflow читает имя пользователя из этих переменных (по убыванию приоритета).
# Перебиваем все, чтобы любой следующий run автоматически шёл от Yulia.
os.environ["LOGNAME"]  = "Yulia"
os.environ["USER"]     = "Yulia"
os.environ["USERNAME"] = "Yulia"

# ---- B. Перезаписать автора у уже залогированных run'ов ----------------
client = MlflowClient()
exp = client.get_experiment_by_name("metro_bus_uplift_v1")
all_runs = client.search_runs(experiment_ids=[exp.experiment_id])

for r in all_runs:
    client.set_tag(r.info.run_id, "mlflow.user", "Yulia")
    print(f"  обновлён run: {r.data.tags.get('mlflow.runName', r.info.run_id[:8])}")

print(f"\n✅ Перезаписан автор у {len(all_runs)} run'ов")
print("✅ Переменные окружения установлены для будущих run'ов")

  обновлён run: baseline_zero_uplift

✅ Перезаписан автор у 1 run'ов
✅ Переменные окружения установлены для будущих run'ов


In [13]:
import xgboost as xgb

# ---- 1. XGBoost с разумными дефолтами ----------------------------------
# Параметры выбраны как "стандартный старт" из документации XGBoost.
# Цель Day 4 — получить рабочую модель и сравнить её с baseline,
# а не выжать максимум. Тюнинг гиперпараметров — Day 5 через Optuna.
xgb_params = {
    "objective":        "reg:squarederror",  # регрессия с MSE-loss
    "n_estimators":     300,                  # число деревьев
    "max_depth":        6,                    # глубина дерева
    "learning_rate":    0.1,                  # шаг обучения
    "subsample":        0.8,                  # доля строк на каждое дерево
    "colsample_bytree": 0.8,                  # доля фичей на каждое дерево
    "random_state":     42,
    "n_jobs":           -1,                   # все ядра CPU
    "tree_method":      "hist",               # быстрый алгоритм для больших данных
}

print("Обучаем XGBoost...")
print(f"  размер train: {X_train.shape}")
print(f"  размер test : {X_test.shape}")
print(f"  параметры   : n_estimators={xgb_params['n_estimators']}, "
      f"max_depth={xgb_params['max_depth']}, lr={xgb_params['learning_rate']}")

t_start = time.time()
xgb_model = xgb.XGBRegressor(**xgb_params)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],   # отслеживаем качество на тесте
    verbose=False,
)
train_time = time.time() - t_start

# ---- 2. Предсказание и метрики -----------------------------------------
t_start = time.time()
y_pred_xgb = xgb_model.predict(X_test)
inference_time = time.time() - t_start

mae_x, rmse_x, mape_x, n_mape_x = calc_metrics(y_test.values, y_pred_xgb)

print("\n" + "=" * 55)
print("XGBoost (default params, no tuning)")
print("=" * 55)
print(f"MAE  : {mae_x:.3f} пасс/час  (baseline: {mae_b:.3f})")
print(f"RMSE : {rmse_x:.3f} пасс/час  (baseline: {rmse_b:.3f})")
print(f"MAPE : {mape_x:.2f}%        (baseline: {mape_b:.2f}%)")
print(f"\nВремя обучения  : {train_time:.1f} сек")
print(f"Время инференса : {inference_time*1000:.1f} мс на {len(X_test):,} строк")
print(f"                  ({inference_time/len(X_test)*1e6:.1f} мкс на 1 предсказание)")

# ---- 3. Улучшение относительно baseline --------------------------------
mae_improvement  = (mae_b - mae_x) / mae_b * 100
rmse_improvement = (rmse_b - rmse_x) / rmse_b * 100
print(f"\n📊 Улучшение MAE относительно baseline : {mae_improvement:+.1f}%")
print(f"📊 Улучшение RMSE относительно baseline: {rmse_improvement:+.1f}%")

# ---- 4. Логирование в MLflow -------------------------------------------
with mlflow.start_run(run_name="xgboost_default"):
    mlflow.log_params({
        "model_type":   "xgboost",
        "tuning":       "none_default_params",
        "n_train":      len(y_train),
        "n_test":       len(y_test),
        "n_features":   X_train.shape[1],
        "train_period": "2024-10-01 .. 2024-11-17",
        "test_period":  "2024-12-01 .. 2024-12-22",
        **{f"hp_{k}": v for k, v in xgb_params.items()},
    })
    mlflow.log_metrics({
        "mae":             mae_x,
        "rmse":            rmse_x,
        "mape":            mape_x if not np.isnan(mape_x) else -1.0,
        "mape_n_samples":  n_mape_x,
        "train_time_sec":  train_time,
        "inference_time_ms": inference_time * 1000,
        "mae_improvement_vs_baseline_pct":  mae_improvement,
        "rmse_improvement_vs_baseline_pct": rmse_improvement,
    })
    # Сохраняем саму модель как артефакт — её потом можно будет загрузить
    mlflow.xgboost.log_model(xgb_model, name="model")
    
    print("\n✅ XGBoost залогирован в MLflow вместе с моделью-артефактом")

Обучаем XGBoost...
  размер train: (836875, 32)
  размер test : (406021, 32)
  параметры   : n_estimators=300, max_depth=6, lr=0.1

XGBoost (default params, no tuning)
MAE  : 10.921 пасс/час  (baseline: 11.678)
RMSE : 23.514 пасс/час  (baseline: 26.330)
MAPE : 132.22%        (baseline: 100.00%)

Время обучения  : 37.0 сек
Время инференса : 743.3 мс на 406,021 строк
                  (1.8 мкс на 1 предсказание)

📊 Улучшение MAE относительно baseline : +6.5%
📊 Улучшение RMSE относительно baseline: +10.7%


TypeError: log_model() missing 1 required positional argument: 'artifact_path'

In [14]:
# ---- Удаляем failed-run и логируем заново с правильным API -------------
client = MlflowClient()
exp = client.get_experiment_by_name("metro_bus_uplift_v1")

# Находим failed-run от XGBoost и удаляем
failed_runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="tags.mlflow.runName = 'xgboost_default' and attributes.status = 'FAILED'",
)
for r in failed_runs:
    client.delete_run(r.info.run_id)
    print(f"🗑️  Удалён failed-run: {r.info.run_id[:8]}")

# Логируем заново — модель и метрики уже в памяти, не пересчитываем
with mlflow.start_run(run_name="xgboost_default"):
    mlflow.log_params({
        "model_type":   "xgboost",
        "tuning":       "none_default_params",
        "n_train":      len(y_train),
        "n_test":       len(y_test),
        "n_features":   X_train.shape[1],
        "train_period": "2024-10-01 .. 2024-11-17",
        "test_period":  "2024-12-01 .. 2024-12-22",
        **{f"hp_{k}": v for k, v in xgb_params.items()},
    })
    mlflow.log_metrics({
        "mae":             mae_x,
        "rmse":            rmse_x,
        "mape":            mape_x if not np.isnan(mape_x) else -1.0,
        "mape_n_samples":  n_mape_x,
        "train_time_sec":  train_time,
        "inference_time_ms": inference_time * 1000,
        "mae_improvement_vs_baseline_pct":  mae_improvement,
        "rmse_improvement_vs_baseline_pct": rmse_improvement,
    })
    # В MLflow 2.18 параметр называется artifact_path, а не name
    mlflow.xgboost.log_model(xgb_model, artifact_path="model")

print("\n✅ XGBoost залогирован чисто, с моделью-артефактом")

🗑️  Удалён failed-run: 4a65830a


2026/04/25 20:06:24 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



✅ XGBoost залогирован чисто, с моделью-артефактом


In [15]:
from sklearn.ensemble import RandomForestRegressor

# ---- 1. Random Forest с разумными дефолтами ----------------------------
# Параметры подобраны как разумный компромисс между качеством и временем
# обучения на 836k строк. n_estimators=100 — стандарт sklearn,
# max_depth=15 ограничивает рост деревьев (без ограничения RF на 800k
# строк может занять 30+ минут и переобучиться).
rf_params = {
    "n_estimators":     100,
    "max_depth":        15,
    "min_samples_split": 20,    # минимум 20 примеров для разделения узла
    "min_samples_leaf":  10,    # минимум 10 примеров в листе
    "max_features":     "sqrt", # стандарт для RF: sqrt(n_features)
    "random_state":     42,
    "n_jobs":           -1,     # все ядра CPU
}

print("Обучаем Random Forest...")
print(f"  размер train: {X_train.shape}")
print(f"  параметры   : n_estimators={rf_params['n_estimators']}, "
      f"max_depth={rf_params['max_depth']}")
print("  ⏳ Это займёт 3-8 минут на 836k строк, наберись терпения...")

t_start = time.time()
rf_model = RandomForestRegressor(**rf_params)
rf_model.fit(X_train, y_train)
train_time_rf = time.time() - t_start

# ---- 2. Предсказание и метрики -----------------------------------------
t_start = time.time()
y_pred_rf = rf_model.predict(X_test)
inference_time_rf = time.time() - t_start

mae_r, rmse_r, mape_r, n_mape_r = calc_metrics(y_test.values, y_pred_rf)

print("\n" + "=" * 55)
print("Random Forest (default params, no tuning)")
print("=" * 55)
print(f"MAE  : {mae_r:.3f} пасс/час  (baseline: {mae_b:.3f}, xgb: {mae_x:.3f})")
print(f"RMSE : {rmse_r:.3f} пасс/час  (baseline: {rmse_b:.3f}, xgb: {rmse_x:.3f})")
print(f"MAPE : {mape_r:.2f}%        (baseline: {mape_b:.2f}%, xgb: {mape_x:.2f}%)")
print(f"\nВремя обучения  : {train_time_rf:.1f} сек ({train_time_rf/60:.1f} мин)")
print(f"Время инференса : {inference_time_rf*1000:.1f} мс на {len(X_test):,} строк")

# ---- 3. Улучшение относительно baseline --------------------------------
mae_improvement_rf  = (mae_b - mae_r) / mae_b * 100
rmse_improvement_rf = (rmse_b - rmse_r) / rmse_b * 100
print(f"\n📊 Улучшение MAE vs baseline : {mae_improvement_rf:+.1f}% "
      f"(xgb: {mae_improvement:+.1f}%)")
print(f"📊 Улучшение RMSE vs baseline: {rmse_improvement_rf:+.1f}% "
      f"(xgb: {rmse_improvement:+.1f}%)")

# ---- 4. Логирование в MLflow -------------------------------------------
with mlflow.start_run(run_name="random_forest_default"):
    mlflow.log_params({
        "model_type":   "random_forest",
        "tuning":       "none_default_params",
        "n_train":      len(y_train),
        "n_test":       len(y_test),
        "n_features":   X_train.shape[1],
        "train_period": "2024-10-01 .. 2024-11-17",
        "test_period":  "2024-12-01 .. 2024-12-22",
        **{f"hp_{k}": v for k, v in rf_params.items()},
    })
    mlflow.log_metrics({
        "mae":             mae_r,
        "rmse":            rmse_r,
        "mape":            mape_r if not np.isnan(mape_r) else -1.0,
        "mape_n_samples":  n_mape_r,
        "train_time_sec":  train_time_rf,
        "inference_time_ms": inference_time_rf * 1000,
        "mae_improvement_vs_baseline_pct":  mae_improvement_rf,
        "rmse_improvement_vs_baseline_pct": rmse_improvement_rf,
    })
    mlflow.sklearn.log_model(rf_model, artifact_path="model")

print("\n✅ Random Forest залогирован в MLflow")

Обучаем Random Forest...
  размер train: (836875, 32)
  параметры   : n_estimators=100, max_depth=15
  ⏳ Это займёт 3-8 минут на 836k строк, наберись терпения...

Random Forest (default params, no tuning)
MAE  : 11.199 пасс/час  (baseline: 11.678, xgb: 10.921)
RMSE : 24.313 пасс/час  (baseline: 26.330, xgb: 23.514)
MAPE : 128.95%        (baseline: 100.00%, xgb: 132.22%)

Время обучения  : 18.3 сек (0.3 мин)
Время инференса : 859.6 мс на 406,021 строк

📊 Улучшение MAE vs baseline : +4.1% (xgb: +6.5%)
📊 Улучшение RMSE vs baseline: +7.7% (xgb: +10.7%)


2026/04/25 20:09:36 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



✅ Random Forest залогирован в MLflow


In [ ]:
import pandas as pd

# ---- Сводная таблица для печати в ноутбук и для Главы 3 ВКР -----------
results_summary = pd.DataFrame([
    {
        "Модель":           "Baseline (predict 0)",
        "MAE":              round(mae_b, 2),
        "RMSE":             round(rmse_b, 2),
        "MAPE_%":           round(mape_b, 1),
        "Δ_MAE_%":          0.0,
        "Время_обуч_сек":   0.0,
        "Время_инф_мс":     0.0,
    },
    {
        "Модель":           "Random Forest",
        "MAE":              round(mae_r, 2),
        "RMSE":             round(rmse_r, 2),
        "MAPE_%":           round(mape_r, 1),
        "Δ_MAE_%":          round(mae_improvement_rf, 1),
        "Время_обуч_сек":   round(train_time_rf, 1),
        "Время_инф_мс":     round(inference_time_rf * 1000, 1),
    },
    {
        "Модель":           "XGBoost",
        "MAE":              round(mae_x, 2),
        "RMSE":             round(rmse_x, 2),
        "MAPE_%":           round(mape_x, 1),
        "Δ_MAE_%":          round(mae_improvement, 1),
        "Время_обуч_сек":   round(train_time, 1),
        "Время_инф_мс":     round(inference_time * 1000, 1),
    },
])

print("=" * 75)
print("СРАВНЕНИЕ МОДЕЛЕЙ — Эксперимент 1 для раздела 3.2 ВКР")
print("=" * 75)
print(results_summary.to_string(index=False))
print("=" * 75)

# Сохраняем в CSV — пригодится для вставки таблицы в ВКР
output_path = project_root / "reports" / "models_comparison_v1.csv"
output_path.parent.mkdir(exist_ok=True)
results_summary.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"\n💾 Таблица сохранена: {output_path}")

# Определяем лучшую модель
best_idx = results_summary["MAE"].idxmin()
best_model = results_summary.loc[best_idx, "Модель"]
best_mae = results_summary.loc[best_idx, "MAE"]
print(f"\n🏆 Лучшая модель по MAE: {best_model} (MAE = {best_mae})")
print("   → её будем тюнить в Day 5 через Optuna")

СРАВНЕНИЕ МОДЕЛЕЙ — Эксперимент 1 для раздела 3.2 ВКР
              Модель   MAE  RMSE  MAPE_%  Δ_MAE_%  Время_обуч_сек  Время_инф_мс
Baseline (predict 0) 11.68 26.33   100.0      0.0             0.0           0.0
       Random Forest 11.20 24.31   128.9      4.1            18.3         859.6
             XGBoost 10.92 23.51   132.2      6.5            37.0         743.3

💾 Таблица сохранена: c:\metro-bus-ml\data\processed\models_comparison_v1.csv

🏆 Лучшая модель по MAE: XGBoost (MAE = 10.92)
   → её будем тюнить в Day 5 через Optuna
